# Compile Season Data

Get tournament matchup matrix for specified season

In [1]:
season = 2024

playin_losers = (  # remove play-in losers from seeding data
    1224,  # Howard
    1438,  # Virginia
    1286,  # Montana St
    1129,  # Boise St
)

model_path = '../data/models/mens/2025_03_02_model.pkl'
data_path = '../data/models/mens/2025_03_02_data.parquet'

season

2024

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\mens_kaggle\tournament_results.parquet')

df = df.loc[df['Season'] == season, :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2024,1101,Abilene Chr,-1.0,-0.333333
1,2024,1102,Air Force,-1.0,-1.000000
2,2024,1103,Akron,-1.0,-0.666667
3,2024,1104,Alabama,2.0,1.333333
4,2024,1105,Alabama A&M,-1.0,-1.000000
...,...,...,...,...,...
375,2024,1476,Stonehill,-1.0,-1.000000
376,2024,1477,East Texas A&M,-1.0,-1.000000
377,2024,1478,Le Moyne,-1.0,-1.000000
378,2024,1479,Mercyhurst,-1.0,-1.000000


### Barttorvik Ratings

In [3]:
df_barttorvik = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik\barttorvik.parquet')

df_barttorvik = df_barttorvik.loc[df_barttorvik['Season'] == season, :].reset_index(drop=True)

df_barttorvik

,Season,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2024,Houston,118.5,85.3,33.2,0.978,0.882353,49.7,44.0,29.9,39.0,13.7,24.7,36.9,30.2,64.2,43.4,30.0,16.1,7.4,49.1,60.2,36.6,40.9,64.3,78.774,2.288,63.130,69.4,32.991
1,2024,Connecticut,126.8,94.1,32.7,0.969,0.911765,57.1,45.1,33.3,32.5,14.9,16.2,36.5,26.8,66.2,43.7,31.9,14.2,8.3,63.7,46.0,40.9,33.2,65.6,81.643,1.710,58.255,74.2,34.029
2,2024,Purdue,126.1,94.3,31.8,0.966,0.878788,56.0,47.7,42.8,23.0,16.5,14.0,37.4,24.7,69.0,48.1,31.4,9.6,6.0,64.5,55.1,35.0,37.2,68.6,83.272,1.854,52.251,72.1,35.470
3,2024,Auburn,120.5,92.1,28.4,0.957,0.794118,54.1,43.4,38.2,41.0,14.9,18.2,32.9,30.3,71.0,42.8,29.8,16.0,8.6,62.0,42.8,37.5,33.3,70.8,80.954,2.198,43.272,75.2,27.793
4,2024,Arizona,121.6,93.3,28.3,0.955,0.757576,55.0,48.7,36.7,25.7,16.1,18.1,36.3,23.1,73.4,47.8,33.4,9.0,8.6,59.1,52.6,32.6,38.2,73.1,81.443,1.979,76.327,71.9,30.384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,2024,Stonehill,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
358,2024,Saint Francis,93.3,117.3,-24.0,0.067,0.214286,47.2,53.0,32.6,35.4,21.2,17.1,32.9,31.3,66.4,52.9,35.4,10.1,11.8,52.6,52.0,35.1,37.1,66.4,79.692,0.327,0.200,60.6,8.567
359,2024,IU Indy,92.7,116.9,-24.2,0.065,0.103448,46.5,58.2,33.2,33.4,21.3,18.5,30.0,35.5,68.6,59.0,38.0,6.0,7.7,42.4,55.3,24.3,37.5,68.3,79.492,2.044,4.750,72.3,10.790
360,2024,Coppin St.,84.7,110.0,-25.3,0.047,0.068966,42.1,51.3,31.1,38.3,22.9,21.8,27.0,38.6,67.3,51.0,34.5,8.0,9.5,40.7,58.6,34.4,37.6,67.3,80.172,1.292,4.358,72.6,9.758


In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\MTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 1192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1173,youngstown st.,1464
1174,youngstown state,1464
1175,youngstown-st,1464
1176,youngstown-state,1464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1178

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_7024\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Cal St. Bakersfield,cal state bakersfield,92
5,Southeast Missouri St.,southeast missouri state,93
6,Mississippi Valley St.,mississippi valley state,93
7,Texas A&M Corpus Chris,texas a&m-corpus christi,96
8,UC Riverside,uc riverside,100
9,Green Bay,green bay,100


In [8]:
df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik

,Season,TeamID,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2024,1222,Houston,118.5,85.3,33.2,0.978,0.882353,49.7,44.0,29.9,39.0,13.7,24.7,36.9,30.2,64.2,43.4,30.0,16.1,7.4,49.1,60.2,36.6,40.9,64.3,78.774,2.288,63.130,69.4,32.991
1,2024,1163,Connecticut,126.8,94.1,32.7,0.969,0.911765,57.1,45.1,33.3,32.5,14.9,16.2,36.5,26.8,66.2,43.7,31.9,14.2,8.3,63.7,46.0,40.9,33.2,65.6,81.643,1.710,58.255,74.2,34.029
2,2024,1345,Purdue,126.1,94.3,31.8,0.966,0.878788,56.0,47.7,42.8,23.0,16.5,14.0,37.4,24.7,69.0,48.1,31.4,9.6,6.0,64.5,55.1,35.0,37.2,68.6,83.272,1.854,52.251,72.1,35.470
3,2024,1120,Auburn,120.5,92.1,28.4,0.957,0.794118,54.1,43.4,38.2,41.0,14.9,18.2,32.9,30.3,71.0,42.8,29.8,16.0,8.6,62.0,42.8,37.5,33.3,70.8,80.954,2.198,43.272,75.2,27.793
4,2024,1112,Arizona,121.6,93.3,28.3,0.955,0.757576,55.0,48.7,36.7,25.7,16.1,18.1,36.3,23.1,73.4,47.8,33.4,9.0,8.6,59.1,52.6,32.6,38.2,73.1,81.443,1.979,76.327,71.9,30.384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,2024,1476,Stonehill,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
358,2024,1383,Saint Francis,93.3,117.3,-24.0,0.067,0.214286,47.2,53.0,32.6,35.4,21.2,17.1,32.9,31.3,66.4,52.9,35.4,10.1,11.8,52.6,52.0,35.1,37.1,66.4,79.692,0.327,0.200,60.6,8.567
359,2024,1237,IU Indy,92.7,116.9,-24.2,0.065,0.103448,46.5,58.2,33.2,33.4,21.3,18.5,30.0,35.5,68.6,59.0,38.0,6.0,7.7,42.4,55.3,24.3,37.5,68.3,79.492,2.044,4.750,72.3,10.790
360,2024,1164,Coppin St.,84.7,110.0,-25.3,0.047,0.068966,42.1,51.3,31.1,38.3,22.9,21.8,27.0,38.6,67.3,51.0,34.5,8.0,9.5,40.7,58.6,34.4,37.6,67.3,80.172,1.292,4.358,72.6,9.758


In [9]:
df = pd.merge(
    df,
    df_barttorvik.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832
376,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953
377,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904
378,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Barttorvik Previous Seasons

In [10]:
df_barttorvik_prev = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik_full_season\barttorvik_full_season.parquet')

df_barttorvik_prev = df_barttorvik_prev.loc[df_barttorvik_prev['Season'] == season, :].reset_index(drop=True)

df_barttorvik_prev

,Season,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2024,Abilene Christian,0.442,-2.137,0.588750,3.158750
1,2024,Air Force,0.567,2.424,0.371000,-5.150750
2,2024,Akron,0.648,5.612,0.649000,5.652500
3,2024,Alabama,0.955,27.053,0.884250,19.621500
4,2024,Alabama A&M,0.212,-11.586,0.130000,-16.677000
...,...,...,...,...,...,...
362,2024,Wright St.,0.445,-2.010,0.569750,2.691000
363,2024,Wyoming,0.536,1.330,0.540750,1.777250
364,2024,Xavier,0.889,19.535,0.835500,14.911500
365,2024,Yale,0.753,9.970,0.684333,7.184667


In [11]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik_prev['TEAM'].unique())

df_match.head(25)

  0%|          | 0/367 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Winston Salem St.,winston-salem-state,91
5,Cal St. Bakersfield,cal state bakersfield,92
6,Southeast Missouri St.,southeast missouri state,93
7,Mississippi Valley St.,mississippi valley state,93
8,Texas A&M Corpus Chris,texas a&m-corpus christi,96
9,Rice,rice,100


In [12]:
df_barttorvik_prev.insert(1, 'TeamID', df_barttorvik_prev['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik_prev

,Season,TeamID,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2024,1101,Abilene Christian,0.442,-2.137,0.588750,3.158750
1,2024,1102,Air Force,0.567,2.424,0.371000,-5.150750
2,2024,1103,Akron,0.648,5.612,0.649000,5.652500
3,2024,1104,Alabama,0.955,27.053,0.884250,19.621500
4,2024,1105,Alabama A&M,0.212,-11.586,0.130000,-16.677000
...,...,...,...,...,...,...,...
362,2024,1460,Wright St.,0.445,-2.010,0.569750,2.691000
363,2024,1461,Wyoming,0.536,1.330,0.540750,1.777250
364,2024,1462,Xavier,0.889,19.535,0.835500,14.911500
365,2024,1463,Yale,0.753,9.970,0.684333,7.184667


In [13]:
df = pd.merge(
    df,
    df_barttorvik_prev.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.loc[df['Past Year BARTHAG'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
8,2024,1109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
226,2024,1327,Okla City,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### My Rankings

In [15]:
df_rankings = pd.read_parquet(fr'..\data\preprocessed\mens_my_rankings\my_rankings_{season}.parquet')

df_rankings.insert(0, 'Season', season)

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,Purdue,2.748541,0.270535,1.226961,0.956426,68.431771
1,2024,Connecticut,2.630943,0.293531,1.235397,0.941866,66.215369
2,2024,Houston,2.563600,0.316181,1.169363,0.853182,64.703058
3,2024,Tennessee,2.469288,0.239553,1.150460,0.910907,69.831425
4,2024,Auburn,2.462638,0.271007,1.193665,0.922657,70.519650
...,...,...,...,...,...,...,...
357,2024,Virginia Military Institute,-2.001527,-0.230590,0.887748,1.118338,75.151420
358,2024,Stonehill,-2.056838,-0.211164,0.917356,1.128521,68.998300
359,2024,Coppin State,-2.101066,-0.248831,0.852367,1.101197,67.573169
360,2024,Mississippi Valley State,-2.626128,-0.323167,0.851673,1.174840,65.665662


In [16]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Purdue,purdue,100
2,Southern Utah,southern utah,100
3,Binghamton,binghamton,100
4,Grambling,grambling,100
5,Western Illinois,western illinois,100
6,Portland State,portland state,100
7,Northern Illinois,northern illinois,100
8,Cal State Fullerton,cal state fullerton,100
9,Louisville,louisville,100


In [17]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,1345,Purdue,2.748541,0.270535,1.226961,0.956426,68.431771
1,2024,1163,Connecticut,2.630943,0.293531,1.235397,0.941866,66.215369
2,2024,1222,Houston,2.563600,0.316181,1.169363,0.853182,64.703058
3,2024,1397,Tennessee,2.469288,0.239553,1.150460,0.910907,69.831425
4,2024,1120,Auburn,2.462638,0.271007,1.193665,0.922657,70.519650
...,...,...,...,...,...,...,...,...
357,2024,1440,Virginia Military Institute,-2.001527,-0.230590,0.887748,1.118338,75.151420
358,2024,1476,Stonehill,-2.056838,-0.211164,0.917356,1.128521,68.998300
359,2024,1164,Coppin State,-2.101066,-0.248831,0.852367,1.101197,67.573169
360,2024,1290,Mississippi Valley State,-2.626128,-0.323167,0.851673,1.174840,65.665662


In [18]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [19]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.032447,-0.035603,1.008106,1.043709,69.448098
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.308710,-0.085275,1.047912,1.133187,63.100239
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.456864,0.053497,1.061477,1.007980,67.054397
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.193852,0.220932,1.230582,1.009651,73.737463
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-1.034594,-0.144498,0.933596,1.078094,71.074148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.056838,-0.211164,0.917356,1.128521,68.998300
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.151632,-0.162850,0.938564,1.101414,67.364372
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.141824,-0.083781,1.003072,1.086853,68.314842
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
8,2024,1109,Alliant Intl,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Starters

In [21]:
df_starters = pd.read_parquet(fr'..\data\preprocessed\mens_starters\starters_{season}.parquet')

df_starters.insert(0, 'Season', season)

df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

df_starters

,Season,Team,Starters
0,2024,Connecticut,0.548104
1,2024,Purdue,0.536821
2,2024,Houston,0.496831
3,2024,Auburn,0.483800
4,2024,North Carolina,0.441960
...,...,...,...
357,2024,Stonehill,-0.400817
358,2024,Virginia Military Institute,-0.406939
359,2024,Buffalo,-0.422366
360,2024,Mississippi Valley State,-0.442076


In [22]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Connecticut,connecticut,100
2,Miami (OH),miami (oh),100
3,East Carolina,east carolina,100
4,Saint Louis,saint louis,100
5,Cal State Bakersfield,cal state bakersfield,100
6,Columbia,columbia,100
7,Binghamton,binghamton,100
8,Gardner-Webb,gardner webb,100
9,Canisius,canisius,100


In [23]:
df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

df_starters

,Season,TeamID,Team,Starters
0,2024,1163,Connecticut,0.548104
1,2024,1345,Purdue,0.536821
2,2024,1222,Houston,0.496831
3,2024,1120,Auburn,0.483800
4,2024,1314,North Carolina,0.441960
...,...,...,...,...
357,2024,1476,Stonehill,-0.400817
358,2024,1440,Virginia Military Institute,-0.406939
359,2024,1138,Buffalo,-0.422366
360,2024,1290,Mississippi Valley State,-0.442076


In [24]:
df_starters.loc[df_starters['TeamID'].isna(), :]

,Season,TeamID,Team,Starters


In [25]:
df = pd.merge(
    df,
    df_starters.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.032447,-0.035603,1.008106,1.043709,69.448098,0.063427
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.308710,-0.085275,1.047912,1.133187,63.100239,-0.191559
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.456864,0.053497,1.061477,1.007980,67.054397,0.234113
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.193852,0.220932,1.230582,1.009651,73.737463,0.283066
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-1.034594,-0.144498,0.933596,1.078094,71.074148,-0.074870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.056838,-0.211164,0.917356,1.128521,68.998300,-0.400817
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.151632,-0.162850,0.938564,1.101414,67.364372,-0.111816
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.141824,-0.083781,1.003072,1.086853,68.314842,-0.070744
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df.loc[df['Starters'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
8,2024,1109,Alliant Intl,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Openskill Ratings

In [27]:
df_os = pd.read_parquet(fr'..\data\preprocessed\mens_os_rankings\os_rankings_{season}.parquet')

df_os.insert(0, 'Season', season)

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2024,Connecticut,51.086743,38.595099
1,2024,Houston,50.517837,38.093567
2,2024,Iowa State,48.187216,36.334127
3,2024,Purdue,49.322721,36.220631
4,2024,Auburn,47.461159,35.736290
...,...,...,...,...
357,2024,IUPUI,1.774340,-12.158040
358,2024,Virginia Military Institute,2.390031,-12.570655
359,2024,Detroit Mercy,1.112006,-13.481287
360,2024,Coppin State,0.370521,-13.662552


In [28]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Boston University,boston university,100
2,Tennessee State,tennessee state,100
3,North Florida,north florida,100
4,Cal State Fullerton,cal state fullerton,100
5,Texas Southern,texas southern,100
6,Mount St. Mary's,mount st. mary's,100
7,Middle Tennessee,middle tennessee,100
8,Canisius,canisius,100
9,Radford,radford,100


In [29]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2024,1163,Connecticut,51.086743,38.595099
1,2024,1222,Houston,50.517837,38.093567
2,2024,1235,Iowa State,48.187216,36.334127
3,2024,1345,Purdue,49.322721,36.220631
4,2024,1120,Auburn,47.461159,35.736290
...,...,...,...,...,...
357,2024,1237,IUPUI,1.774340,-12.158040
358,2024,1440,Virginia Military Institute,2.390031,-12.570655
359,2024,1178,Detroit Mercy,1.112006,-13.481287
360,2024,1164,Coppin State,0.370521,-13.662552


In [30]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.032447,-0.035603,1.008106,1.043709,69.448098,0.063427,24.504253,12.252205
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.308710,-0.085275,1.047912,1.133187,63.100239,-0.191559,16.689471,3.046381
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.456864,0.053497,1.061477,1.007980,67.054397,0.234113,31.015647,17.953905
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.193852,0.220932,1.230582,1.009651,73.737463,0.283066,40.127889,28.383919
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-1.034594,-0.144498,0.933596,1.078094,71.074148,-0.074870,14.001525,1.578976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.056838,-0.211164,0.917356,1.128521,68.998300,-0.400817,2.053793,-11.474784
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.151632,-0.162850,0.938564,1.101414,67.364372,-0.111816,12.329532,-0.371308
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.141824,-0.083781,1.003072,1.086853,68.314842,-0.070744,16.213913,3.887557
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
8,2024,1109,Alliant Intl,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,1118,Armstrong St,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,1121,Augusta,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,1128,Birmingham So,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,1134,Brooklyn,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,1147,Centenary,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,1215,Hardin-Simmons,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2024,1216,Hartford,-1.0,-0.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042,-28.007,0.26400,-11.57475,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,2024,1289,Morris Brown,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2024,1302,NE Illinois,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Betting Odds

In [32]:
df_bo = pd.read_parquet('../data/preprocessed/mens_betting/betting.parquet')

df_bo = df_bo.loc[df_bo['Season'] == season, :].reset_index(drop=True)

df_bo

,Season,Team,Implied Champion Probability
0,2024,Connecticut,0.222222
1,2024,Purdue,0.133333
2,2024,Alabama,0.024390
3,2024,NC State,0.004975
4,2024,Tennessee,0.062500
...,...,...,...
63,2024,Montana State,0.000500
64,2024,South Dakota State,0.000500
65,2024,St Peter's,0.000500
66,2024,Stetson,0.000500


In [33]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_bo['Team'].unique())

df_match.head(25)

  0%|          | 0/68 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Connecticut,connecticut,100
1,St Mary's,st. mary's,100
2,Wisconsin,wisconsin,100
3,Florida Atlantic,florida atlantic,100
4,Texas Tech,texas tech,100
5,Mississippi State,mississippi state,100
6,New Mexico,new mexico,100
7,TCU,tcu,100
8,Nebraska,nebraska,100
9,Nevada,nevada,100


In [34]:
df_bo.insert(1, 'TeamID', df_bo['Team'].map(team_to_spelling).map(spelling_to_id))

df_bo

,Season,TeamID,Team,Implied Champion Probability
0,2024,1163,Connecticut,0.222222
1,2024,1345,Purdue,0.133333
2,2024,1104,Alabama,0.024390
3,2024,1301,NC State,0.004975
4,2024,1397,Tennessee,0.062500
...,...,...,...,...
63,2024,1286,Montana State,0.000500
64,2024,1355,South Dakota State,0.000500
65,2024,1389,St Peter's,0.000500
66,2024,1391,Stetson,0.000500


In [35]:
df = pd.merge(
    df,
    df_bo.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.032447,-0.035603,1.008106,1.043709,69.448098,0.063427,24.504253,12.252205,NaN
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.308710,-0.085275,1.047912,1.133187,63.100239,-0.191559,16.689471,3.046381,NaN
2,2024,1103,Akron,-1.0,-0.666667,105.7,101.9,3.8,0.604,0.687500,52.0,48.6,33.6,29.0,17.2,16.6,29.5,25.6,66.7,50.7,30.0,8.0,7.8,48.2,45.4,41.5,36.8,66.7,79.972,2.582,11.000,72.4,11.844,0.648,5.612,0.64900,5.65250,0.456864,0.053497,1.061477,1.007980,67.054397,0.234113,31.015647,17.953905,0.000999
3,2024,1104,Alabama,2.0,1.333333,125.6,101.4,24.2,0.921,0.656250,56.3,49.9,35.2,39.6,16.0,15.6,34.9,29.9,74.2,51.1,31.9,10.2,12.5,51.5,45.1,46.8,36.7,73.5,82.460,2.102,27.770,78.4,36.310,0.955,27.053,0.88425,19.62150,2.193852,0.220932,1.230582,1.009651,73.737463,0.283066,40.127889,28.383919,0.024390
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-1.034594,-0.144498,0.933596,1.078094,71.074148,-0.074870,14.001525,1.578976,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.056838,-0.211164,0.917356,1.128521,68.998300,-0.400817,2.053793,-11.474784,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.151632,-0.162850,0.938564,1.101414,67.364372,-0.111816,12.329532,-0.371308,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.141824,-0.083781,1.003072,1.086853,68.314842,-0.070744,16.213913,3.887557,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df.loc[df['Implied Champion Probability'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability
0,2024,1101,Abilene Chr,-1.0,-0.333333,101.0,104.6,-3.6,0.401,0.454545,47.1,51.6,37.9,39.9,17.7,20.2,25.9,30.4,70.6,53.0,32.5,5.7,8.0,48.6,47.2,26.4,31.6,69.4,78.977,2.163,4.892,73.1,13.772,0.442,-2.137,0.58875,3.15875,0.032447,-0.035603,1.008106,1.043709,69.448098,0.063427,24.504253,12.252205,NaN
1,2024,1102,Air Force,-1.0,-1.000000,105.6,111.0,-5.4,0.360,0.290323,53.8,54.2,29.6,39.1,18.8,17.6,23.8,31.5,63.4,53.7,36.6,11.8,11.3,62.0,49.9,47.2,36.7,62.9,79.000,1.570,0.200,68.2,20.073,0.567,2.424,0.37100,-5.15075,-0.308710,-0.085275,1.047912,1.133187,63.100239,-0.191559,16.689471,3.046381,NaN
4,2024,1105,Alabama A&M,-1.0,-1.000000,93.1,108.9,-15.8,0.142,0.352941,45.8,48.9,47.7,45.9,22.5,19.6,31.9,32.4,71.9,46.7,35.1,10.9,8.3,45.3,53.8,26.7,37.1,71.5,79.781,1.878,0.200,72.0,9.918,0.212,-11.586,0.13000,-16.67700,-1.034594,-0.144498,0.933596,1.078094,71.074148,-0.074870,14.001525,1.578976,NaN
5,2024,1106,Alabama St,-1.0,-1.000000,92.6,104.3,-11.7,0.203,0.344828,41.0,49.1,28.5,43.4,16.3,20.0,31.7,29.1,69.1,47.6,34.1,11.1,12.5,44.4,53.3,38.4,41.0,68.1,80.237,1.891,0.402,71.9,13.689,0.100,-19.276,0.10300,-19.29750,-0.940350,-0.117734,0.916806,1.034540,69.000954,-0.195720,10.341737,-2.513072,NaN
6,2024,1107,SUNY Albany,-1.0,-1.000000,103.6,110.2,-6.6,0.330,0.387097,50.4,51.4,29.2,38.9,17.7,17.0,30.6,29.4,74.4,54.6,29.3,7.4,11.5,44.5,46.6,36.9,29.6,74.0,78.852,1.711,0.344,72.5,11.654,0.123,-18.178,0.23875,-10.81025,-0.514868,-0.040804,1.048196,1.089000,73.589454,-0.153456,17.442745,5.047938,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376,2024,1476,Stonehill,-1.0,-1.000000,90.5,113.7,-23.2,0.068,0.100000,46.7,52.7,22.6,29.4,19.5,16.6,22.5,31.0,69.4,51.7,35.9,7.2,13.9,51.7,60.8,47.0,46.2,69.1,79.889,1.711,0.200,69.7,12.832,0.181,-13.249,NaN,NaN,-2.056838,-0.211164,0.917356,1.128521,68.998300,-0.400817,2.053793,-11.474784,NaN
377,2024,1477,East Texas A&M,-1.0,-1.000000,94.3,111.4,-17.1,0.128,0.333333,46.0,52.4,30.8,39.2,16.7,18.3,24.3,33.1,67.9,52.4,35.0,11.4,11.3,52.6,51.0,46.0,30.4,67.1,78.070,1.804,2.490,69.2,14.953,0.197,-13.003,NaN,NaN,-1.151632,-0.162850,0.938564,1.101414,67.364372,-0.111816,12.329532,-0.371308,NaN
378,2024,1478,Le Moyne,-1.0,-1.000000,98.9,111.2,-12.3,0.206,0.413793,50.1,50.6,25.3,27.8,16.6,17.8,23.1,29.9,68.5,51.0,33.4,8.7,8.1,60.7,56.3,47.5,43.0,68.3,78.762,2.502,0.200,76.5,7.904,NaN,NaN,NaN,NaN,-1.141824,-0.083781,1.003072,1.086853,68.314842,-0.070744,16.213913,3.887557,NaN
379,2024,1479,Mercyhurst,-1.0,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Map to Matchups

In [37]:
# df_seeds = pd.read_csv(fr'..\data\unprocessed\kaggle\{season}_tourney_seeds.csv')

# df_seeds = df_seeds.loc[df_seeds['Tournament'] == 'M', :].reset_index(drop=True)

# df_seeds.rename(columns={'Seed': 'Region Seed'}, inplace=True)
# df_seeds.insert(2, 'Region', df_seeds['Region Seed'].str[0])
# df_seeds.insert(3, 'Seed', df_seeds['Region Seed'].str.extract('(\d+)').astype(int))

# df_seeds

In [38]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

df_seeds = df_seeds.loc[~df_seeds['TeamID'].isin(playin_losers), :].reset_index(drop=True)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2024,1,W,False,1163
1,2024,2,W,False,1235
2,2024,3,W,False,1228
3,2024,4,W,False,1120
4,2024,5,W,False,1361
...,...,...,...,...,...
59,2024,12,Z,False,1241
60,2024,13,Z,False,1436
61,2024,14,Z,False,1324
62,2024,15,Z,False,1443


In [39]:
id_to_region = dict(zip(df_seeds['TeamID'], df_seeds['Region']))
id_to_seed = dict(zip(df_seeds['TeamID'], df_seeds['Seed']))

df_mod = pd.DataFrame(
    [
        (team_a, team_b) 
        for team_a in df_seeds['TeamID'].unique() 
        for team_b in df_seeds['TeamID'].unique() 
        if team_a != team_b
    ],
    columns=['Team A ID', 'Team B ID']
)

df_mod.insert(0, 'Season', season)
df_mod['Team A Region'] = df_mod['Team A ID'].map(id_to_region)
df_mod['Team B Region'] = df_mod['Team B ID'].map(id_to_region)
df_mod['Team A Seed'] = df_mod['Team A ID'].map(id_to_seed)
df_mod['Team B Seed'] = df_mod['Team B ID'].map(id_to_seed)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2024,1163,1235,W,W,1,2
1,2024,1163,1228,W,W,1,3
2,2024,1163,1120,W,W,1,4
3,2024,1163,1361,W,W,1,5
4,2024,1163,1140,W,W,1,6
...,...,...,...,...,...,...,...
4027,2024,1255,1301,Z,Z,16,11
4028,2024,1255,1241,Z,Z,16,12
4029,2024,1255,1436,Z,Z,16,13
4030,2024,1255,1324,Z,Z,16,14


Calculate round of matchup

In [40]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # no play-in games in this data

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0       False
1       False
2       False
3       False
4       False
        ...  
4027    False
4028    False
4029    False
4030    False
4031    False
Length: 4032, dtype: bool

In [41]:
df_mod['Round'] = -1

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round
0,2024,1163,1235,W,W,1,2,4
1,2024,1163,1228,W,W,1,3,4
2,2024,1163,1120,W,W,1,4,3
3,2024,1163,1361,W,W,1,5,3
4,2024,1163,1140,W,W,1,6,4
...,...,...,...,...,...,...,...,...
4027,2024,1255,1301,Z,Z,16,11,4
4028,2024,1255,1241,Z,Z,16,12,3
4029,2024,1255,1436,Z,Z,16,13,3
4030,2024,1255,1324,Z,Z,16,14,4


Get Head-to-Head

In [42]:
df_h2h = pd.read_parquet('../data/preprocessed/mens_h2h/h2h.parquet')

df_h2h = df_h2h.loc[df_h2h['Season'] == season, :].reset_index(drop=True)

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2024,Abilene Christian,Air Force,NaN,0.960669
1,2024,Abilene Christian,Alabama,NaN,-1.276758
2,2024,Abilene Christian,Alabama A&M,NaN,0.117263
3,2024,Abilene Christian,Alabama State,NaN,-1.253728
4,2024,Abilene Christian,Alcorn State,NaN,-0.187852
...,...,...,...,...,...
73793,2024,Youngstown State,William & Mary,NaN,1.438673
73794,2024,Youngstown State,Winthrop,NaN,0.006061
73795,2024,Youngstown State,Wisconsin,NaN,-0.111978
73796,2024,Youngstown State,Wright State,0.791622,-0.187724


In [43]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Texas A&M-Commerce,tx a&m commerce,91
1,Radford,radford,100
2,Quinnipiac,quinnipiac,100
3,Queens (NC),Queens (NC),100
4,Purdue Fort Wayne,purdue fort wayne,100
5,Purdue,purdue,100
6,Providence,providence,100
7,Princeton,princeton,100
8,Presbyterian,presbyterian,100
9,Rhode Island,rhode island,100


In [44]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2024,1101,Abilene Christian,1102,Air Force,NaN,0.960669
1,2024,1101,Abilene Christian,1104,Alabama,NaN,-1.276758
2,2024,1101,Abilene Christian,1105,Alabama A&M,NaN,0.117263
3,2024,1101,Abilene Christian,1106,Alabama State,NaN,-1.253728
4,2024,1101,Abilene Christian,1108,Alcorn State,NaN,-0.187852
...,...,...,...,...,...,...,...
73793,2024,1464,Youngstown State,1456,William & Mary,NaN,1.438673
73794,2024,1464,Youngstown State,1457,Winthrop,NaN,0.006061
73795,2024,1464,Youngstown State,1458,Wisconsin,NaN,-0.111978
73796,2024,1464,Youngstown State,1460,Wright State,0.791622,-0.187724


In [45]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps
0,2024,1163,1235,W,W,1,2,4,NaN,-0.297288
1,2024,1163,1228,W,W,1,3,4,NaN,0.846631
2,2024,1163,1120,W,W,1,4,3,NaN,0.000000
3,2024,1163,1361,W,W,1,5,3,NaN,0.041428
4,2024,1163,1140,W,W,1,6,4,NaN,-0.706294
...,...,...,...,...,...,...,...,...,...,...
4027,2024,1255,1301,Z,Z,16,11,4,NaN,-0.396836
4028,2024,1255,1241,Z,Z,16,12,3,NaN,-0.647769
4029,2024,1255,1436,Z,Z,16,13,3,NaN,NaN
4030,2024,1255,1324,Z,Z,16,14,4,NaN,0.138418


Get team names

In [46]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\MTeams.csv')

df_teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [47]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps
0,2024,1163,Connecticut,1235,Iowa St,W,W,1,2,4,NaN,-0.297288
1,2024,1163,Connecticut,1228,Illinois,W,W,1,3,4,NaN,0.846631
2,2024,1163,Connecticut,1120,Auburn,W,W,1,4,3,NaN,0.000000
3,2024,1163,Connecticut,1361,San Diego St,W,W,1,5,3,NaN,0.041428
4,2024,1163,Connecticut,1140,BYU,W,W,1,6,4,NaN,-0.706294
...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,1255,Longwood,1301,NC State,Z,Z,16,11,4,NaN,-0.396836
4028,2024,1255,Longwood,1241,James Madison,Z,Z,16,12,3,NaN,-0.647769
4029,2024,1255,Longwood,1436,Vermont,Z,Z,16,13,3,NaN,NaN
4030,2024,1255,Longwood,1324,Oakland,Z,Z,16,14,4,NaN,0.138418


Map features

In [48]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

df_features['Team A ADJ OE Team B ADJ DE'] = team_a_features['ADJ OE'] + team_b_features['ADJ DE']
df_features['Team B ADJ OE Team A ADJ DE'] = team_b_features['ADJ OE'] + team_a_features['ADJ DE']

df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A BARTHAG'] = team_a_features['BARTHAG']
df_features['Team B BARTHAG'] = team_b_features['BARTHAG']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,6.0,1.666667,13.4,6.6,6.8,0.017,0.117647,5.2,-2.0,-2.8,-2.7,-0.8,-9.5,4.9,-2.6,-2.0,-3.2,0.4,3.4,-0.4,5.7,-12.9,8.9,-11.8,-2.9,1.075,-0.118,13.917,4.5,2.340,0.096,14.112,0.15775,10.69375,0.423871,0.029670,0.099277,0.069607,-2.181198,0.165282,2.899527,2.260973,0.169591,214.3,207.5,2.107656,2.077986,0.969,0.952
1,6.0,1.333333,2.5,-6.6,9.1,0.051,0.147059,3.3,-2.9,-4.5,5.4,-0.2,3.9,0.3,0.7,-4.3,-2.9,-2.4,6.0,0.0,19.3,3.4,2.6,5.3,-5.2,0.993,-0.729,1.133,-0.3,2.176,0.142,16.727,0.00150,1.16325,0.630596,0.079539,0.017655,-0.061884,-4.839806,0.138353,6.093587,5.366882,0.189964,227.5,218.4,2.239147,2.159608,0.969,0.918
2,5.0,1.666667,6.3,2.0,4.3,0.012,0.117647,3.0,1.7,-4.9,-8.5,0.0,-2.0,3.6,-3.5,-4.8,0.9,2.1,-1.8,-0.3,1.7,3.2,3.4,-0.1,-5.2,0.689,-0.488,14.983,-1.0,6.236,0.081,12.045,0.03100,3.64450,0.168305,0.022524,0.041732,0.019208,-4.304281,0.064304,3.625584,2.858810,0.166667,218.9,214.6,2.158054,2.135530,0.969,0.957
3,1.0,0.333333,15.0,-0.3,15.3,0.094,0.224265,7.3,-2.1,-4.2,-0.8,-1.1,-1.8,3.8,-1.2,-0.8,-4.6,1.4,2.3,-0.6,12.9,-2.6,3.1,-7.3,-1.5,0.862,-0.794,15.218,1.3,4.351,0.044,9.024,-0.01500,0.78325,0.414142,0.109496,0.116906,0.007410,-0.944347,0.216719,10.526202,10.338266,0.212321,221.2,205.9,2.169853,2.060356,0.969,0.875
4,7.0,2.666667,6.6,-4.4,11.0,0.061,0.214795,2.0,-2.9,8.0,0.4,-0.3,0.2,3.7,2.1,-3.8,-4.5,0.1,6.5,1.2,0.5,-1.2,-9.8,-0.9,-4.4,0.449,-0.520,38.705,0.4,4.380,0.206,20.023,0.05775,5.20550,1.089422,0.081268,0.052637,-0.028632,-4.409225,0.300552,11.354623,10.770125,0.207297,225.3,214.3,2.205894,2.124626,0.969,0.908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,-1.0,0.000000,-10.0,3.5,-13.5,-0.317,-0.030466,-1.3,0.8,7.0,4.2,4.3,1.7,8.1,-3.8,-0.9,1.4,-0.3,-2.1,-0.5,1.9,0.9,-5.2,4.2,-1.1,-0.906,-0.111,-30.798,-4.6,-20.235,-0.310,-13.863,-0.39300,-16.48600,-1.522706,-0.093794,-0.070023,0.023772,-1.195287,-0.164079,-11.185037,-12.623651,-0.004475,204.3,217.8,2.068779,2.162573,0.483,0.800
4028,0.0,0.333333,-8.4,3.5,-11.9,-0.290,-0.328446,-5.0,4.6,5.3,1.0,3.3,-0.2,4.7,-2.1,-3.1,2.1,6.0,-0.4,-1.7,-6.3,7.3,-9.8,4.4,-3.0,-0.175,-0.105,13.737,-1.7,0.210,-0.129,-4.748,-0.07750,-2.98025,-1.291733,-0.113307,-0.069212,0.044094,-2.862444,-0.407242,-15.549989,-14.112203,-0.002349,204.3,216.2,2.048456,2.161763,0.483,0.773
4029,-1.0,-0.333333,-1.8,4.4,-6.2,-0.169,-0.231855,-2.4,4.6,9.8,10.2,4.2,4.5,14.3,2.2,2.4,4.5,3.1,-3.7,-3.4,-4.1,6.1,-15.8,3.0,3.5,0.422,0.017,13.278,-3.5,-0.581,-0.170,-6.659,-0.30425,-12.02000,-0.845393,-0.058706,-0.010113,0.048593,3.223311,-0.287066,-13.520449,-13.306831,-0.000499,203.4,209.6,2.043958,2.102664,0.483,0.652
4030,0.0,0.333333,-4.6,-0.9,-3.7,-0.100,-0.095825,-2.7,0.7,8.6,9.3,1.8,3.3,6.1,-3.1,-0.2,0.4,0.7,-0.7,-0.8,-5.3,-13.1,-12.5,-1.2,0.1,0.808,-0.260,10.233,-7.0,-5.519,0.255,10.221,0.02875,0.98150,-0.649591,-0.021677,-0.026054,-0.004377,-0.209106,-0.209034,-5.507647,-6.256362,-0.000499,208.7,212.4,2.096927,2.118604,0.483,0.583


In [49]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2024,1163,Connecticut,1235,Iowa St,W,W,1,2,4,NaN,-0.297288,6.0,1.666667,13.4,6.6,6.8,0.017,0.117647,5.2,-2.0,-2.8,-2.7,-0.8,-9.5,4.9,-2.6,-2.0,-3.2,0.4,3.4,-0.4,5.7,-12.9,8.9,-11.8,-2.9,1.075,-0.118,13.917,4.5,2.340,0.096,14.112,0.15775,10.69375,0.423871,0.029670,0.099277,0.069607,-2.181198,0.165282,2.899527,2.260973,0.169591,214.3,207.5,2.107656,2.077986,0.969,0.952
1,2024,1163,Connecticut,1228,Illinois,W,W,1,3,4,NaN,0.846631,6.0,1.333333,2.5,-6.6,9.1,0.051,0.147059,3.3,-2.9,-4.5,5.4,-0.2,3.9,0.3,0.7,-4.3,-2.9,-2.4,6.0,0.0,19.3,3.4,2.6,5.3,-5.2,0.993,-0.729,1.133,-0.3,2.176,0.142,16.727,0.00150,1.16325,0.630596,0.079539,0.017655,-0.061884,-4.839806,0.138353,6.093587,5.366882,0.189964,227.5,218.4,2.239147,2.159608,0.969,0.918
2,2024,1163,Connecticut,1120,Auburn,W,W,1,4,3,NaN,0.000000,5.0,1.666667,6.3,2.0,4.3,0.012,0.117647,3.0,1.7,-4.9,-8.5,0.0,-2.0,3.6,-3.5,-4.8,0.9,2.1,-1.8,-0.3,1.7,3.2,3.4,-0.1,-5.2,0.689,-0.488,14.983,-1.0,6.236,0.081,12.045,0.03100,3.64450,0.168305,0.022524,0.041732,0.019208,-4.304281,0.064304,3.625584,2.858810,0.166667,218.9,214.6,2.158054,2.135530,0.969,0.957
3,2024,1163,Connecticut,1361,San Diego St,W,W,1,5,3,NaN,0.041428,1.0,0.333333,15.0,-0.3,15.3,0.094,0.224265,7.3,-2.1,-4.2,-0.8,-1.1,-1.8,3.8,-1.2,-0.8,-4.6,1.4,2.3,-0.6,12.9,-2.6,3.1,-7.3,-1.5,0.862,-0.794,15.218,1.3,4.351,0.044,9.024,-0.01500,0.78325,0.414142,0.109496,0.116906,0.007410,-0.944347,0.216719,10.526202,10.338266,0.212321,221.2,205.9,2.169853,2.060356,0.969,0.875
4,2024,1163,Connecticut,1140,BYU,W,W,1,6,4,NaN,-0.706294,7.0,2.666667,6.6,-4.4,11.0,0.061,0.214795,2.0,-2.9,8.0,0.4,-0.3,0.2,3.7,2.1,-3.8,-4.5,0.1,6.5,1.2,0.5,-1.2,-9.8,-0.9,-4.4,0.449,-0.520,38.705,0.4,4.380,0.206,20.023,0.05775,5.20550,1.089422,0.081268,0.052637,-0.028632,-4.409225,0.300552,11.354623,10.770125,0.207297,225.3,214.3,2.205894,2.124626,0.969,0.908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,1255,Longwood,1301,NC State,Z,Z,16,11,4,NaN,-0.396836,-1.0,0.000000,-10.0,3.5,-13.5,-0.317,-0.030466,-1.3,0.8,7.0,4.2,4.3,1.7,8.1,-3.8,-0.9,1.4,-0.3,-2.1,-0.5,1.9,0.9,-5.2,4.2,-1.1,-0.906,-0.111,-30.798,-4.6,-20.235,-0.310,-13.863,-0.39300,-16.48600,-1.522706,-0.093794,-0.070023,0.023772,-1.195287,-0.164079,-11.185037,-12.623651,-0.004475,204.3,217.8,2.068779,2.162573,0.483,0.800
4028,2024,1255,Longwood,1241,James Madison,Z,Z,16,12,3,NaN,-0.647769,0.0,0.333333,-8.4,3.5,-11.9,-0.290,-0.328446,-5.0,4.6,5.3,1.0,3.3,-0.2,4.7,-2.1,-3.1,2.1,6.0,-0.4,-1.7,-6.3,7.3,-9.8,4.4,-3.0,-0.175,-0.105,13.737,-1.7,0.210,-0.129,-4.748,-0.07750,-2.98025,-1.291733,-0.113307,-0.069212,0.044094,-2.862444,-0.407242,-15.549989,-14.112203,-0.002349,204.3,216.2,2.048456,2.161763,0.483,0.773
4029,2024,1255,Longwood,1436,Vermont,Z,Z,16,13,3,NaN,NaN,-1.0,-0.333333,-1.8,4.4,-6.2,-0.169,-0.231855,-2.4,4.6,9.8,10.2,4.2,4.5,14.3,2.2,2.4,4.5,3.1,-3.7,-3.4,-4.1,6.1,-15.8,3.0,3.5,0.422,0.017,13.278,-3.5,-0.581,-0.170,-6.659,-0.30425,-12.02000,-0.845393,-0.058706,-0.010113,0.048593,3.223311,-0.287066,-13.520449,-13.306831,-0.000499,203.4,209.6,2.043958,2.102664,0.483,0.652
4030,202

In [50]:
df_mod.insert(1, 'Round', df_mod.pop('Round'))

df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Round,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Implied Champion Probability,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2024,4,1163,Connecticut,1235,Iowa St,NaN,-0.297288,6.0,1.666667,13.4,6.6,6.8,0.017,0.117647,5.2,-2.0,-2.8,-2.7,-0.8,-9.5,4.9,-2.6,-2.0,-3.2,0.4,3.4,-0.4,5.7,-12.9,8.9,-11.8,-2.9,1.075,-0.118,13.917,4.5,2.340,0.096,14.112,0.15775,10.69375,0.423871,0.029670,0.099277,0.069607,-2.181198,0.165282,2.899527,2.260973,0.169591,214.3,207.5,2.107656,2.077986,0.969,0.952
1,2024,4,1163,Connecticut,1228,Illinois,NaN,0.846631,6.0,1.333333,2.5,-6.6,9.1,0.051,0.147059,3.3,-2.9,-4.5,5.4,-0.2,3.9,0.3,0.7,-4.3,-2.9,-2.4,6.0,0.0,19.3,3.4,2.6,5.3,-5.2,0.993,-0.729,1.133,-0.3,2.176,0.142,16.727,0.00150,1.16325,0.630596,0.079539,0.017655,-0.061884,-4.839806,0.138353,6.093587,5.366882,0.189964,227.5,218.4,2.239147,2.159608,0.969,0.918
2,2024,3,1163,Connecticut,1120,Auburn,NaN,0.000000,5.0,1.666667,6.3,2.0,4.3,0.012,0.117647,3.0,1.7,-4.9,-8.5,0.0,-2.0,3.6,-3.5,-4.8,0.9,2.1,-1.8,-0.3,1.7,3.2,3.4,-0.1,-5.2,0.689,-0.488,14.983,-1.0,6.236,0.081,12.045,0.03100,3.64450,0.168305,0.022524,0.041732,0.019208,-4.304281,0.064304,3.625584,2.858810,0.166667,218.9,214.6,2.158054,2.135530,0.969,0.957
3,2024,3,1163,Connecticut,1361,San Diego St,NaN,0.041428,1.0,0.333333,15.0,-0.3,15.3,0.094,0.224265,7.3,-2.1,-4.2,-0.8,-1.1,-1.8,3.8,-1.2,-0.8,-4.6,1.4,2.3,-0.6,12.9,-2.6,3.1,-7.3,-1.5,0.862,-0.794,15.218,1.3,4.351,0.044,9.024,-0.01500,0.78325,0.414142,0.109496,0.116906,0.007410,-0.944347,0.216719,10.526202,10.338266,0.212321,221.2,205.9,2.169853,2.060356,0.969,0.875
4,2024,4,1163,Connecticut,1140,BYU,NaN,-0.706294,7.0,2.666667,6.6,-4.4,11.0,0.061,0.214795,2.0,-2.9,8.0,0.4,-0.3,0.2,3.7,2.1,-3.8,-4.5,0.1,6.5,1.2,0.5,-1.2,-9.8,-0.9,-4.4,0.449,-0.520,38.705,0.4,4.380,0.206,20.023,0.05775,5.20550,1.089422,0.081268,0.052637,-0.028632,-4.409225,0.300552,11.354623,10.770125,0.207297,225.3,214.3,2.205894,2.124626,0.969,0.908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,4,1255,Longwood,1301,NC State,NaN,-0.396836,-1.0,0.000000,-10.0,3.5,-13.5,-0.317,-0.030466,-1.3,0.8,7.0,4.2,4.3,1.7,8.1,-3.8,-0.9,1.4,-0.3,-2.1,-0.5,1.9,0.9,-5.2,4.2,-1.1,-0.906,-0.111,-30.798,-4.6,-20.235,-0.310,-13.863,-0.39300,-16.48600,-1.522706,-0.093794,-0.070023,0.023772,-1.195287,-0.164079,-11.185037,-12.623651,-0.004475,204.3,217.8,2.068779,2.162573,0.483,0.800
4028,2024,3,1255,Longwood,1241,James Madison,NaN,-0.647769,0.0,0.333333,-8.4,3.5,-11.9,-0.290,-0.328446,-5.0,4.6,5.3,1.0,3.3,-0.2,4.7,-2.1,-3.1,2.1,6.0,-0.4,-1.7,-6.3,7.3,-9.8,4.4,-3.0,-0.175,-0.105,13.737,-1.7,0.210,-0.129,-4.748,-0.07750,-2.98025,-1.291733,-0.113307,-0.069212,0.044094,-2.862444,-0.407242,-15.549989,-14.112203,-0.002349,204.3,216.2,2.048456,2.161763,0.483,0.773
4029,2024,3,1255,Longwood,1436,Vermont,NaN,NaN,-1.0,-0.333333,-1.8,4.4,-6.2,-0.169,-0.231855,-2.4,4.6,9.8,10.2,4.2,4.5,14.3,2.2,2.4,4.5,3.1,-3.7,-3.4,-4.1,6.1,-15.8,3.0,3.5,0.422,0.017,13.278,-3.5,-0.581,-0.170,-6.659,-0.30425,-12.02000,-0.845393,-0.058706,-0.010113,0.048593,3.223311,-0.287066,-13.520449,-13.306831,-0.000499,203.4,209.6,2.043958,2.102664,0.483,0.652
4030,2024,4,1255,Longwood,1324,Oakland,NaN,0.138418,0.0,0.333333,-4.6,-0.9,-3.7,-0.100,-0.095825,-2.7,0.7,8.6,9.3,1.8,3.3,6.1,-3.1,-0.2,0.4,0.7,-0

Check that data follows same format as the data that the model was trained on

In [51]:
df_mod_training = pd.read_parquet(data_path)

assert all(df_mod_training.drop(columns=['Result']).columns == df_mod.columns), 'Columns do not match'

'Columns Match'

'Columns Match'

### Get Model Predictions

In [52]:
import pickle

with open(model_path, 'rb') as f:
    mod = pickle.load(f)

mod

LGBMClassifier(early_stopping_round=25, feature_fraction=0.06289161158581992,
               lambda_l1=1.0860375709395227, lambda_l2=0.897726181746411,
               max_depth=6, metric='rmse', min_child_samples=69,
               monotone_constraints=[0, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1, -1, -1,
                                     -1, -1, 0, 1, 0, 0, -1, -1, 1, -1, 0, 0, 0,
                                     0, 0, 1, -1, ...],
               n_estimators=500, num_leaves=10, random_state=22, verbosity=-1)

In [53]:
X = df_mod.drop(columns=['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B'])

predictions = mod.predict_proba(X)[:, 1]

predictions

array([0.76203259, 0.77159131, 0.68919963, ..., 0.13865069, 0.40316493,
       0.47515794])

Turn predictions into matchup matrix

In [54]:
df_matrix = (
    df_mod[['Team A ID', 'Team B ID']]
    .assign(Prediction=predictions)
    .pivot(
        index=['Team A ID'], 
        columns=['Team B ID'],
        values='Prediction',
    )
)

df_matrix

Team B ID,1103,1104,1112,1120,1124,1140,1155,1158,1159,1160,1161,1163,1166,1173,1179,1181,1182,1194,1196,1211,1212,1213,1222,1228,1235,1241,1242,1246,1253,1255,1266,1270,1277,1280,1287,1301,1304,1305,1307,1314,1321,1324,1332,1345,1355,1359,1361,1376,1388,1389,1391,1395,1397,1400,1401,1403,1412,1429,1436,1443,1447,1450,1458,1463
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1103,NaN,0.091424,0.100999,0.049055,0.131083,0.155765,0.157105,0.367367,0.560826,0.250313,0.225430,0.034348,0.106627,0.222755,0.281165,0.106767,0.312563,0.233548,0.209328,0.103026,0.730086,0.286865,0.038950,0.145166,0.104024,0.310438,0.104804,0.121994,0.535948,0.704541,0.093873,0.338311,0.109939,0.171521,0.399835,0.205099,0.225470,0.198172,0.193089,0.073454,0.255970,0.519174,0.248627,0.059206,0.592316,0.430386,0.085425,0.234464,0.135958,0.557775,0.670341,0.179640,0.063735,0.137210,0.229399,0.168736,0.419211,0.303317,0.441591,0.543127,0.786664,0.163285,0.222902,0.306252
1104,0.887449,NaN,0.495728,0.446144,0.446509,0.629222,0.734520,0.844120,0.816294,0.698888,0.663397,0.213377,0.484626,0.662964,0.728947,0.465467,0.819684,0.733430,0.670293,0.370357,0.902744,0.796515,0.290977,0.499721,0.630285,0.781615,0.477631,0.528642,0.870246,0.902599,0.557683,0.802345,0.615933,0.674149,0.836165,0.776244,0.650679,0.773625,0.654538,0.514926,0.705286,0.895274,0.728910,0.324404,0.910130,0.845026,0.548337,0.759642,0.495212,0.884772,0.899043,0.691232,0.523457,0.581930,0.638315,0.611653,0.843289,0.747786,0.840188,0.878728,0.892666,0.690447,0.621297,0.802720
1112,0.920101,0.512363,NaN,0.582447,0.569483,0.693065,0.695271,0.848028,0.888044,0.704243,0.764256,0.277244,0.489642,0.742488,0.812140,0.626449,0.848107,0.679434,0.731327,0.434969,0.944352,0.837338,0.263829,0.560484,0.564288,0.816304,0.691762,0.631711,0.939420,0.925273,0.638145,0.787844,0.629192,0.720149,0.888539,0.811626,0.790048,0.844721,0.730414,0.522722,0.740460,0.908033,0.739081,0.516795,0.910863,0.853979,0.559995,0.840035,0.639849,0.858673,0.907419,0.715965,0.567121,0.669379,0.772259,0.688352,0.762712,0.792121,0.858367,0.894852,0.917132,0.715442,0.762986,0.864187
1120,0.942205,0.542500,0.517429,NaN,0.587811,0.718672,0.727100,0.875870,0.943376,0.752414,0.789643,0.341665,0.510303,0.760715,0.840927,0.522076,0.881595,0.802651,0.770356,0.510027,0.945687,0.877556,0.285309,0.664446,0.640003,0.872160,0.661159,0.600889,0.923330,0.953046,0.555464,0.870865,0.709740,0.783636,0.901448,0.832055,0.802074,0.847943,0.822159,0.463446,0.751709,0.905288,0.751350,0.363941,0.939845,0.899116,0.735807,0.782864,0.602511,0.940083,0.935136,0.794294,0.514761,0.681920,0.819617,0.734574,0.882190,0.824634,0.878228,0.943663,0.935917,0.807095,0.736274,0.761602
1124,0.841726,0.574906,0.447245,0.403862,NaN,0.664542,0.652738,0.766535,0.882888,0.677107,0.677627,0.257938,0.449187,0.579388,0.820853,0.362534,0.780075,0.671490,0.565665,0.450448,0.899802,0.627877,0.231932,0.444465,0.512421,0.738477,0.555774,0.449446,0.834585,0.909486,0.473802,0.765529,0.557811,0.651269,0.840664,0.634145,0.612767,0.669080,0.631254,0.422350,0.607154,0.886073,0.695656,0.308807,0.862892,0.781507,0.532749,0.645180,0.471677,0.869053,0.866036,0.626406,0.506122,0.573884,0.559854,0.647553,0.817842,0.725458,0.744863,0.815890,0.892299,0.637932,0.619721,0.805674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1443,0.419586,0.103249,0.087278,0.049238,0.158699,0.115367,0.192579,0.309076,0.480313,0.174399,0.177746,0.032347,0.067406,0.152848,0.235006,0.105894,0.390444,0.198300,0.130172,0.127512,0.743769,0.238720,0.034079,0.110393,0.091894,0.287335,0.074378,0.116196,0.566913,0.495737,0.090055,0.420332,0.083192,0.199054,0.496692,0.308922,0.132460,0.170303,0.113852,0.096708,0.211048,0.618824,0.208622,0.061947,0.503129,0.318194,0.096513,0.199685,0.099107,0.505639,0

In [55]:
df_matrix_display = df_matrix.copy()

df_matrix_display.columns = df_matrix_display.columns.map(id_to_team)
df_matrix_display.index = df_matrix_display.index.map(id_to_team)

df_matrix_display

Team B ID,Akron,Alabama,Arizona,Auburn,Baylor,BYU,Clemson,Col Charleston,Colgate,Colorado,Colorado St,Connecticut,Creighton,Dayton,Drake,Duke,Duquesne,FL Atlantic,Florida,Gonzaga,Grambling,Grand Canyon,Houston,Illinois,Iowa St,James Madison,Kansas,Kentucky,Long Beach St,Longwood,Marquette,McNeese St,Michigan St,Mississippi St,Morehead St,NC State,Nebraska,Nevada,New Mexico,North Carolina,Northwestern,Oakland,Oregon,Purdue,S Dakota St,Samford,San Diego St,South Carolina,St Mary's CA,St Peter's,Stetson,TCU,Tennessee,Texas,Texas A&M,Texas Tech,UAB,Utah St,Vermont,WKU,Wagner,Washington St,Wisconsin,Yale
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Akron,NaN,0.091424,0.100999,0.049055,0.131083,0.155765,0.157105,0.367367,0.560826,0.250313,0.225430,0.034348,0.106627,0.222755,0.281165,0.106767,0.312563,0.233548,0.209328,0.103026,0.730086,0.286865,0.038950,0.145166,0.104024,0.310438,0.104804,0.121994,0.535948,0.704541,0.093873,0.338311,0.109939,0.171521,0.399835,0.205099,0.225470,0.198172,0.193089,0.073454,0.255970,0.519174,0.248627,0.059206,0.592316,0.430386,0.085425,0.234464,0.135958,0.557775,0.670341,0.179640,0.063735,0.137210,0.229399,0.168736,0.419211,0.303317,0.441591,0.543127,0.786664,0.163285,0.222902,0.306252
Alabama,0.887449,NaN,0.495728,0.446144,0.446509,0.629222,0.734520,0.844120,0.816294,0.698888,0.663397,0.213377,0.484626,0.662964,0.728947,0.465467,0.819684,0.733430,0.670293,0.370357,0.902744,0.796515,0.290977,0.499721,0.630285,0.781615,0.477631,0.528642,0.870246,0.902599,0.557683,0.802345,0.615933,0.674149,0.836165,0.776244,0.650679,0.773625,0.654538,0.514926,0.705286,0.895274,0.728910,0.324404,0.910130,0.845026,0.548337,0.759642,0.495212,0.884772,0.899043,0.691232,0.523457,0.581930,0.638315,0.611653,0.843289,0.747786,0.840188,0.878728,0.892666,0.690447,0.621297,0.802720
Arizona,0.920101,0.512363,NaN,0.582447,0.569483,0.693065,0.695271,0.848028,0.888044,0.704243,0.764256,0.277244,0.489642,0.742488,0.812140,0.626449,0.848107,0.679434,0.731327,0.434969,0.944352,0.837338,0.263829,0.560484,0.564288,0.816304,0.691762,0.631711,0.939420,0.925273,0.638145,0.787844,0.629192,0.720149,0.888539,0.811626,0.790048,0.844721,0.730414,0.522722,0.740460,0.908033,0.739081,0.516795,0.910863,0.853979,0.559995,0.840035,0.639849,0.858673,0.907419,0.715965,0.567121,0.669379,0.772259,0.688352,0.762712,0.792121,0.858367,0.894852,0.917132,0.715442,0.762986,0.864187
Auburn,0.942205,0.542500,0.517429,NaN,0.587811,0.718672,0.727100,0.875870,0.943376,0.752414,0.789643,0.341665,0.510303,0.760715,0.840927,0.522076,0.881595,0.802651,0.770356,0.510027,0.945687,0.877556,0.285309,0.664446,0.640003,0.872160,0.661159,0.600889,0.923330,0.953046,0.555464,0.870865,0.709740,0.783636,0.901448,0.832055,0.802074,0.847943,0.822159,0.463446,0.751709,0.905288,0.751350,0.363941,0.939845,0.899116,0.735807,0.782864,0.602511,0.940083,0.935136,0.794294,0.514761,0.681920,0.819617,0.734574,0.882190,0.824634,0.878228,0.943663,0.935917,0.807095,0.736274,0.761602
Baylor,0.841726,0.574906,0.447245,0.403862,NaN,0.664542,0.652738,0.766535,0.882888,0.677107,0.677627,0.257938,0.449187,0.579388,0.820853,0.362534,0.780075,0.671490,0.565665,0.450448,0.899802,0.627877,0.231932,0.444465,0.512421,0.738477,0.555774,0.449446,0.834585,0.909486,0.473802,0.765529,0.557811,0.651269,0.840664,0.634145,0.612767,0.669080,0.631254,0.422350,0.607154,0.886073,0.695656,0.308807,0.862892,0.781507,0.532749,0.645180,0.471677,0.869053,0.866036,0.626406,0.506122,0.573884,0.559854,0.647553,0.817842,0.725458,0.744863,0.815890,0.892299,0.637932,0.619721,0.805674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
WKU,0.419586,0.103249,0.087278,0.049238,0.158699,0.115367,0.192579,0.309076,0.480313,0.174399,0.177746,0.032347,0.067406,0.152848,0.235006,0.105894,0.390444,0.198300,0.

In [ ]:
df_matrix.to_csv(f'../data/preprocessed/mens/matchup_matrix_{season}.csv', index=True)
df_matrix_display.to_csv(f'../data/preprocessed/mens/matchup_matrix_display_{season}.csv', index=True)

'Done'

'Done'